### Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

* Tracking agent behavior with logging, analytics, and debugging.
* Transforming prompts, tool selection, and output formatting.
* Adding retries, fallbacks, and early termination logic.
* Applying rate limits, guardrails, and PII detection.

In [11]:
# from openai.types.responses import response
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:openai/gpt-oss-20b")


### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent=create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [13]:

### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [14]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='0be2bbf4-7190-4dc4-bbc7-62220cec8485'), AIMessage(content='4', additional_kwargs={'reasoning_content': 'The user asks: "What is 2+2?" The answer is 4. But the user might expect a short answer. There\'s no special instructions. So respond: "4".'}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 78, 'total_tokens': 127, 'completion_time': 0.051575972, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.004371561, 'prompt_tokens_details': None, 'queue_time': 0.26588364, 'total_time': 0.055947533}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_1074f9ce08', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0a4c8-ce82-7af0-a0fb-3409499104f1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 49, 'total_tokens': 127,

### token Size

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [16]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~187 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='07c53dad-993f-48a9-809c-9997369aa61e'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Find hotels in Paris". We need to use the provided tool: functions.search_hotels. We\'ll call it with city="Paris". Then we need to respond with the results. The tool returns a long response. We should call the function.', 'tool_calls': [{'id': 'fc_989f976d-74b6-4b86-89e5-2f8f74995a00', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 129, 'total_tokens': 207, 'completion_time': 0.099981712, 'completion_tokens_details': {'reasoning_tokens': 54}, 'prompt_time': 0.008238497, 'prompt_tokens_details': None, 'queue_time': 0.382132051, 'total_time': 0.108220209}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e23fc99

KeyboardInterrupt: 

### Fraction

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 131072  # gpt-oss-20B context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~573 tokens (0.4372%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='1d04fef3-9faf-4c0f-b171-895b8cce13ea'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call search_hotels with city Paris.', 'tool_calls': [{'id': 'fc_80c07f4c-cbd4-4b8b-b72c-891860f1b5ab', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 120, 'total_tokens': 156, 'completion_time': 0.037306234, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.005797068, 'prompt_tokens_details': None, 'queue_time': 0.241993381, 'total_time': 0.043103302}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_37c9245f64', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0a4c6-f83d-77f0-81fb-542e92dfffac-0', tool_calls=[{'name': 'sea

### Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [20]:
agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
        interrupt_on={
            "send_email_tool":{
                "allowed_decisions":["approve","reject","edit"]
            },
            "read_email_tool":False,
        }
    )
    ]
)

In [21]:
config={"configurable":{"thread_id":"test-approve"}}
# step1 : Request
result=agent.invoke(
    {"messages":[HumanMessage(content="send email to john@test.com with the subject 'Hello' and body 'How are you?' ")]},
    config=config
)

In [22]:

result

{'messages': [HumanMessage(content="send email to john@test.com with the subject 'Hello' and body 'How are you?' ", additional_kwargs={}, response_metadata={}, id='a06a377d-b009-4009-b9ba-f7e06dc471e1'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the send_email_tool function. Provide JSON with body, recipient, subject.', 'tool_calls': [{'id': 'fc_5677e742-3d45-4a1f-bbf2-f04fbfdceb2b', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 176, 'total_tokens': 233, 'completion_time': 0.065741439, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.008523316, 'prompt_tokens_details': None, 'queue_time': 0.302674452, 'total_time': 0.074264755}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a4315eb300', 'service_tier': 'on_demand', 'finish_reason': 

In [23]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: ✅ Email sent to john@test.com with subject “Hello”.


### Reject

In [24]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [25]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I’m ready to send the email, but I’ll double‑check the details first. Is the recipient “john@test.com”, the subject “Hello”, and the body “How are you?” correct? If so, just let me know and I’ll send it.


In [26]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='a925383f-960b-42e0-bd88-23388bca00ab'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use send_email_tool.', 'tool_calls': [{'id': 'fc_f32cac97-152e-4190-9b68-e21ab8ba0762', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 174, 'total_tokens': 220, 'completion_time': 0.071129515, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.010719462, 'prompt_tokens_details': None, 'queue_time': 0.298148484, 'total_time': 0.081848977}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4f7e7dc26e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='

### Editing

In [33]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [34]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [36]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: I’m sorry, but I can’t send an email to that address.


In [37]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='576aa4f4-ecfb-4bde-9eb3-aea9fa2b0edc'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email to wrong@email.com with subject 'Test' and body 'Hello'. We have a tool send_email_tool. We'll call it.", 'tool_calls': [{'id': 'fc_fb9f9050-dcde-4af8-b144-df24dd01e0c4', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 172, 'total_tokens': 240, 'completion_time': 0.078310374, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.008651252, 'prompt_tokens_details': None, 'queue_time': 0.246111104, 'total_time': 0.086961626}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8b41efc9a3', 'service_tier': 